# cAPTure: fold-aware feature profile

This CPU-only notebook profiles the completed canonical FULL_DEV Parquet artifacts before model preprocessing is frozen. It validates deterministic source/destination TCP-port roles, layered protocol indicators, numeric ranges, categorical code coverage, and fold-training constants. It does not refit anything on validation data, materialize transformed packets, construct graph windows, or train a model.


## 1. Mount Drive and load the project


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_feature_profile.py",
    PROJECT_ROOT / "code/python/tests/test_capture_feature_profile.py",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
    PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml",
    PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("CPU feature-profile environment is ready.")


Mounted at /content/drive
CPU feature-profile environment is ready.


## 2. Run synthetic contract checks


In [2]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_feature_profile.py", "-v"],
    env=test_environment,
    cwd=PROJECT_ROOT,
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/temporalgnn-nids/code/python/tests', '-p', 'test_capture_feature_profile.py', '-v'], returncode=0)

## 3. Configure the FULL_DEV profile

The prepared run is immutable input. This profile writes only a small JSON report and configuration snapshots to a new Drive directory.


In [3]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from utils.capture_feature_profile import run_capture_feature_profile

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs" / "20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 250_000

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_feature_profile"
DRIVE_RUN_DIR = DRIVE_ROOT / "feature_profile_runs" / RUN_ID

print(f"Prepared input: {PREPARED_RUN_DIR}")
print(f"Profile output: {DRIVE_RUN_DIR}")


Prepared input: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Profile output: /content/drive/MyDrive/capture_gate0/feature_profile_runs/20260918T201953_304003Z_feature_profile


## 4. Run and persist the profile


In [4]:
PROFILE = run_capture_feature_profile(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=DRIVE_RUN_DIR,
    batch_size=BATCH_SIZE,
)
print(f"Feature profile saved to {DRIVE_RUN_DIR / 'capture_feature_profile.json'}")


Profiling train_empty_conn...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Profiled train_empty_conn: 1,175,779 packets
Profiling train_qos_mid...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Profiled train_qos_mid: 1,499,717 packets
Profiling train_dollar_char...


...

Profiled train_dollar_char: 2,882,555 packets
Profiling train_slash_char...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Profiled train_slash_char: 6,425,516 packets
Profiling train_sub_exf...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Profiled train_sub_exf: 2,225,807 packets
Feature profile saved to /content/drive/MyDrive/capture_gate0/feature_profile_runs/20260918T201953_304003Z_feature_profile/capture_feature_profile.json


## 5. Review the profile

The validation-coverage tables are diagnostic only. Validation values are never added to a training encoder.


In [5]:
scenario_rows = []
protocol_rows = []
port_rows = []
top_port_rows = []
categorical_rows = []
numeric_rows = []

numeric_features = [
    "frame_length", "ipv4_fragment_offset", "ipv4_length", "ipv4_ttl",
    "ssh_padding_length", "tcp_header_length", "tcp_payload_length",
    "tcp_window_value",
]
categorical_features = [
    "ethernet_type", "ipv4_dscp", "mqtt_connack_reason_code",
    "mqtt_message_type", "mqtt_qos", "mqtt_reserved_flag",
    "mqtt_subscription_qos", "mqtt_version", "ssh_direction",
]

for scenario, profile in PROFILE["scenario_profiles"].items():
    scenario_rows.append({"scenario": scenario, "packets": profile["packets"]})
    protocol_rows.append({"scenario": scenario, **profile["protocol_indicator_counts"]})
    for direction, counts in profile["port_role_counts"].items():
        for role, count in counts.items():
            port_rows.append({
                "scenario": scenario, "direction": direction, "role": role,
                "packets": count, "fraction": count / profile["packets"],
            })
    for direction, counts in profile["top_raw_nonnull_ports"].items():
        top_port_rows.append({
            "scenario": scenario, "direction": direction,
            "top_nonnull_ports": counts,
        })
    for feature in categorical_features:
        stats = profile["feature_statistics"][feature]
        categorical_rows.append({
            "scenario": scenario, "feature": feature,
            "nonnull": stats["nonnull"],
            "null_fraction": stats["null"] / profile["packets"],
            "distinct_nonnull": stats["distinct_nonnull"],
            "value_counts": profile["categorical_counts"][feature],
        })
    for feature in numeric_features:
        stats = profile["feature_statistics"][feature]
        numeric_rows.append({
            "scenario": scenario, "feature": feature,
            "nonnull": stats["nonnull"], "null": stats["null"],
            "minimum": stats["minimum"], "q01": stats["quantiles"]["0.01"],
            "median": stats["quantiles"]["0.5"],
            "q99": stats["quantiles"]["0.99"], "maximum": stats["maximum"],
        })

print("Candidate dimensions")
display(pd.DataFrame([PROFILE["candidate_dimensions"]]))
print("Scenario rows")
display(pd.DataFrame(scenario_rows))
print("Protocol indicator counts")
display(pd.DataFrame(protocol_rows))
print("Fixed TCP-port role distributions")
display(pd.DataFrame(port_rows))
print("Most frequent raw non-null TCP ports")
display(pd.DataFrame(top_port_rows))
print("Categorical-code profiles")
display(pd.DataFrame(categorical_rows))
print("Numeric-magnitude profiles")
display(pd.DataFrame(numeric_rows))

fold_rows = []
coverage_rows = []
for fold, profile in PROFILE["fold_profiles"].items():
    fold_rows.append({
        "fold": fold, "training_packets": profile["training_packets"],
        "training_constants": profile["training_constants"],
        "training_all_missing": profile["training_all_missing"],
    })
    for feature, coverage in profile["categorical_validation_coverage"].items():
        coverage_rows.append({"fold": fold, "feature": feature, **coverage})
print("Fold-training constants")
display(pd.DataFrame(fold_rows))
print("Validation categorical coverage without refitting")
display(pd.DataFrame(coverage_rows))


Candidate dimensions


,shared_across_models,raw_port_values_are_model_features,port_role_features_per_direction,canonical_feature_count,feature_count_after_port_encoding_only,final_feature_count
0,True,False,11,42,62,None


Scenario rows


,scenario,packets
0,train_empty_conn,1175779
1,train_qos_mid,1499717
2,train_dollar_char,2882555
3,train_slash_char,6425516
4,train_sub_exf,2225807


Protocol indicator counts


...

,scenario,feature,nonnull,null,minimum,q01,median,q99,maximum
0,train_empty_conn,frame_length,1175779,0,64,64.000000,73.715759,1518.000000,1518
1,train_empty_conn,ipv4_fragment_offset,1099512,76267,0,0.000000,0.000000,0.000000,2
2,train_empty_conn,ipv4_length,1099512,76267,28,40.000000,55.999989,1500.000000,1500
3,train_empty_conn,ipv4_ttl,1099512,76267,37,38.447609,64.000000,64.000000,64
4,train_empty_conn,ssh_padding_length,6572,1169207,4,4.000000,10.000000,11.000000,11
5,train_empty_conn,tcp_header_length,1098336,77443,20,20.000000,32.000000,32.000000,44
6,train_empty_conn,tcp_payload_length,1098336,77443,0,0.000000,3.997983,1448.000000,1448
7,train_empty_conn,tcp_window_value,1098336,77443,0,0.000000,502.000000,1030.673828,65160
8,train_qos_mid,frame_length,1499717,0,64,64.000000,70.000000,1518.000000,1518
9,train_qos_mid,ipv4_fragment_offset,1426456,73261,0,0.000000,0.000000,0.000000,2


Fold-training constants


,fold,training_packets,training_constants,training_all_missing
0,A,2675496,"[tcp_flag_cwr, tcp_flag_ece, tcp_flag_urg]",[]
1,B,11533878,"[mqtt_version, tcp_flag_cwr, tcp_flag_ece, tcp...",[]


Validation categorical coverage without refitting


,fold,feature,training_distinct_nonnull,validation_distinct_nonnull,validation_unseen_values,validation_unseen_rows,validation_unseen_fraction_of_nonnull,encoder_refit_on_validation
0,A,ethernet_type,3,3,[],0,0.000000e+00,False
1,A,ipv4_dscp,3,3,[],0,0.000000e+00,False
2,A,mqtt_connack_reason_code,2,11,"[158, 170, 173, 24, 245, 47, 70, 9, 97, 99]",113,9.912281e-01,False
3,A,mqtt_message_type,16,16,[],0,0.000000e+00,False
4,A,mqtt_qos,3,4,[3],1,7.781471e-07,False
5,A,mqtt_reserved_flag,14,16,"[11, 9]",16,5.644370e-06,False
6,A,mqtt_subscription_qos,2,2,[],0,0.000000e+00,False
7,A,mqtt_version,2,1,[],0,0.000000e+00,False
8,A,ssh_direction,2,2,[],0,0.000000e+00,False
9,B,ethernet_type,3,3,[],0,0.000000e+00,False
